In [ ]:
import warnings
from pathlib import Path
from matplotlib import pyplot as plt
import prism
import pandas as pd

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
    SharesInflowStocks,
    RestOf
)
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.rest_of import rest_of_preprocessing
from imagematerials.util import reindex_material

In [2]:
# Define the complete timeline, including historic tail

YEAR_START  = 1971 # start year of the simulation period
YEAR_END    = 2100    # end year of the calculations
YEAR_OUT    = 2100    # year of output generation = last year of reporting
time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_OUT, 1)


# Define the scenario list
scenario_list = {"base":("SSP2",["base"]),
                #  "narrow":("SSP2_narrow", ["base", "narrow"]),
                #  "slow":("SSP2_slow",["base", "slow"]),
                #  "close":("SSP2",["base", "close"]),
                #  "narrow_slow_close":("SSP2_narrow_slow_close", ["base", "narrow","slow", "close"])
                }

# Define paths
scenario_base_path = Path("..", "data", "raw")
CP_scenario_path = scenario_base_path/ "IMAGE_CircoMod"
CE_scenario_path = scenario_base_path / 'circular_economy_scenarios'

# define materials list so that arrays have consistent order

materials = [
    'aluminium', 'brick', 'cement', 'cobalt', 'concrete', 'copper', 'glass',        # IF A NEW MATERIAL IS ADDED HERE, add it also in imagematerials.eol.preprocessing
    'lead', 'lithium', 'manganese', 'neodymium', 'nickel', 'plastics',
    'rubber', 'steel', 'tantalum', 'titanium', 'wood'
]

In [3]:
# Run simulations for all scenarios

all_output = {} # to store outputs for all scenarios

# Loop over scenarios
for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running scenario: {climate_scen} from {CP_scenario_path}")
    print(f"Circular economy scenario: {circular_scen}")
    climate_policy_scenario_dir = CP_scenario_path / climate_scen           # path to climate policy scenario data (i.e., from IMAGE)
    circular_economy_scenario_dirs = {
        scenario: CE_scenario_path / scenario for scenario in circular_scen # path to circular economy scenario data
    }

# Get preprocessing data for all sectors and adjust material indices
    
    # BUILDINGS
    bld_sector = get_preprocessing_data("buildings", scenario_base_path, climate_policy_scenario_dir, circular_economy_scenario_dirs)

    #VEHICLES
    vhc_sector = get_preprocessing_data("vehicles", scenario_base_path, climate_policy_scenario_dir, circular_economy_scenario_dirs)

    # #ELECTRICITY
    elc_sector = get_preprocessing_data("electricity", scenario_base_path, climate_policy_scenario_dir, circular_economy_scenario_dirs, cache = False)
    elc_gen, elc_grid_lines, elc_grid_add, elc_stor_phs, elc_stor_other = (
    {sec.name: sec for sec in elc_sector}[k]
    for k in ["elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs", "elc_stor_other"]
)

    # END OF LIFE
    eol_sector = get_preprocessing_data("eol", scenario_base_path, circular_economy_scenario_dirs)
    
    # #REST OF
    rest_sector = rest_of_preprocessing(scenario_base_path, climate_policy_scenario_dir,scenario = climate_scen) #TODO: add this to get_preprocessing_data function
    rest_sector = Sector(name='rest_of', data = rest_sector) # make it a Sector object!

    # Reindex materials in all sectors to ensure consistency
    for sec in [
    elc_gen,
    elc_grid_lines,
    elc_grid_add,
    elc_stor_phs,
    elc_stor_other,
    bld_sector,
    vhc_sector,
    eol_sector,
    rest_sector,
]:
        for key, val in sec.prep_data.items():
            sec.prep_data[key] = reindex_material(val, materials)

# Build and run the model
    factory = ModelFactory(
    [bld_sector, vhc_sector,
    *elc_sector,                # *: unpacks electricity sectors 
     eol_sector, 
     rest_sector,
    ],complete_timeline 
    ).add(GenericStocks, ["buildings", 
                          "vehicles", 
                          "elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs"
                          ]
    ).add(SharesInflowStocks, ["elc_stor_other"]
    ).add(GenericMaterials,  "vehicles"
    ).add(Maintenance, "vehicles"
    ).add(MaterialIntensities, ["buildings",
                        "elc_gen", "elc_grid_lines","elc_grid_add","elc_stor_phs","elc_stor_other"
                        ]
    
    ).add(RestOf, "rest_of", input_sources={
       "gompertz_coefs": "rest_of",
        "gdp_per_capita": "rest_of",
        "population": "rest_of",
        "historic_diff_consumption_mean": "rest_of",
        "historic_diff_consumption_total": "rest_of"
}                  
    ).add(EndOfLife, "eol", input_sources={
    "outflow_by_cohort_materials": ["vehicles", "buildings",
                                    "elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs"
                                    ],
    "inflow_materials":  ["vehicles", "buildings",
                          "elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs"
                         ],
    "collection": "eol",
    "reuse": "eol",
    "recycling": "eol"}
)
    model = factory.finish() # create the model
    

    # Run the simulation (suppressing warnings)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline) 

     # save relevant outputs only to reduce memory usage
    all_output[scen_id] = {
        # MATERIAL inflows, stocks and outflows
        "inflow_materials": [model.vehicles["inflow_materials"], model.buildings["inflow_materials"], 
                             model.elc_gen["inflow_materials"], model.elc_grid_lines["inflow_materials"], model.elc_grid_add["inflow_materials"], model.elc_stor_phs["inflow_materials"], model.elc_stor_other["inflow_materials"]], 
        "stock_by_cohort_materials": [model.vehicles["stock_by_cohort_materials"], model.buildings["stock_by_cohort_materials"], 
                             model.elc_gen["stock_by_cohort_materials"], model.elc_grid_lines["stock_by_cohort_materials"], model.elc_grid_add["stock_by_cohort_materials"], model.elc_stor_phs["stock_by_cohort_materials"], model.elc_stor_other["stock_by_cohort_materials"]], 
        "outflow_by_cohort_materials": [model.vehicles["outflow_by_cohort_materials"], model.buildings["outflow_by_cohort_materials"],
            model.elc_gen["outflow_by_cohort_materials"], model.elc_grid_lines["outflow_by_cohort_materials"], model.elc_grid_add["outflow_by_cohort_materials"], model.elc_stor_phs["outflow_by_cohort_materials"], model.elc_stor_other["outflow_by_cohort_materials"]
            ],
         # PRODUCT inflows, stocks and outflows
        "inflow": [
            model.vehicles["inflow"], model.buildings["inflow"],
            model.elc_gen["inflow"], model.elc_grid_lines["inflow"], model.elc_grid_add["inflow"], model.elc_stor_phs["inflow"], model.elc_stor_other["inflow"]
            ],
        "stocks": [
            model.vehicles["stocks"],model.buildings["stocks"],
            model.elc_gen["stocks"], model.elc_grid_lines["stocks"], model.elc_grid_add["stocks"], model.elc_stor_phs["stocks"], model.elc_stor_other["stocks"]
            ],
        "outflow": [
            model.vehicles["stocks"], 
            model.buildings["stocks"],
            model.elc_gen["stocks"],model.elc_grid_lines["stocks"],model.elc_grid_add["stocks"],model.elc_stor_phs["stocks"],model.elc_stor_other["stocks"]
            ],
        "reusable_materials": model.eol["reusable_materials"],
        "recyclable_materials": model.eol["recyclable_materials"]
    }
    print(f"Finished {scen_id}")

Running scenario: SSP2 from ..\data\raw\IMAGE_CircoMod
Circular economy scenario: ['base']
grid_stock_lines to xarray Dataset
materials_grid_kgperkm to xarray Dataset
lifetime_grid_distr not in conversion_table
grid_stock_add to xarray Dataset
materials_grid_add_kgperunit to xarray Dataset
lifetime_grid_distr not in conversion_table
phs_stock to xarray Dataset
phs_materials to xarray Dataset
phs_lifetime_distr not in conversion_table
oth_storage_stock to xarray Dataset
oth_storage_lifetime_distr not in conversion_table
oth_storage_shares to xarray Dataset
Finished base
